# 🎨 face2sketch — Phase 1: pix2pix GAN

**Photo → Realistic Sketch using pix2pix (Isola et al. 2017).**

| Step | Time |
|---|---|
| Download data + unzip | ~1 min |
| Training (200 epochs) | ~1.5 hr T4/L4 |
| Generate samples | ~10s |

### Architecture
- **Generator:** U-Net (~37M params) — encoder-decoder with skip connections
- **Discriminator:** PatchGAN (~2.8M params) — 70×70 patch classification
- **Loss:** L1 reconstruction (λ=100) + adversarial loss
- **Dataset:** 322 paired (photo, sketch) from CUHK + SKSF-A

In [ ]:
# @title 1. Clone Repo + Install
!git clone https://github.com/weseegod/face2sketch.git /content/face2sketch 2>/dev/null
%cd /content/face2sketch
!git pull origin main

!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q tqdm matplotlib pillow pyyaml einops
!mkdir -p checkpoints samples

import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print('\n✅  Setup complete. Run cells in order.')


In [ ]:
# @title 2. Download Data from Drive (~1 min)
from google.colab import drive
drive.mount('/content/drive')

# Copy data.zip from Drive
!cp /content/drive/MyDrive/face2sketch/data.zip /content/face2sketch/ 2>/dev/null
!unzip -o data.zip -d data/

# Verify
import os
photos = len(os.listdir('data/dataset/photos'))
sketches = len(os.listdir('data/dataset/sketches'))
print(f"Photos: {photos}  |  Sketches: {sketches}  |  Paired: {min(photos, sketches)}")

if photos == 0 or sketches == 0:
    print("\n❌  Data not found! Upload data.zip to Drive: MyDrive/face2sketch/data.zip")
else:
    print(f"\n✅  {min(photos, sketches)} paired images ready for training.")


In [ ]:
# @title 3. Quick Test (100 steps, ~2 min)
assert torch.cuda.is_available(), "⚠️  Enable GPU: Runtime → Change runtime type → T4 GPU"

%cd /content/face2sketch

# First verify the pipeline works with a quick overfit-batch test
!python src/train.py --overfit-batch --device cuda

print("\n✅  Pipeline verified. Ready for full training.")


In [ ]:
# @title 4. Train — 200 epochs (~1.5 hr T4/L4)
assert torch.cuda.is_available(), "⚠️  Enable GPU: Runtime → Change runtime type → T4 GPU"

%cd /content/face2sketch
import glob

# Check for existing checkpoint to resume
latest = sorted(glob.glob("checkpoints/pix2pix_epoch_*.pt"))
if latest:
    print(f"📂  Resuming from {latest[-1]}")
    !python src/train.py --mode train --resume {latest[-1]} --device cuda
else:
    print("🚀  Training from scratch (200 epochs)")
    !python src/train.py --mode train --device cuda --name pix2pix_

print("\n✅  Training complete → checkpoints/pix2pix_best.pt")


In [ ]:
# @title 5. Generate Samples from Validation Set
%cd /content/face2sketch
import os, glob

# Find best checkpoint
ckpt = 'checkpoints/pix2pix_best.pt'
if not os.path.exists(ckpt):
    candidates = sorted(glob.glob('checkpoints/pix2pix_*.pt'), reverse=True)
    for c in candidates:
        if os.path.getsize(c) > 100000:
            ckpt = c; break

if not os.path.exists(ckpt):
    print('❌  No checkpoint found.')
else:
    print(f'📸  Generating samples from: {ckpt}\n')
    
    # Generate single image from a test photo
    !python src/sample.py --checkpoint {ckpt} \
        --input data/dataset/photos/cuhk_f-005-01.jpg \
        --output-dir outputs/ --device cuda
    
    print('\n✅  Sample saved to outputs/')
    
    # Show the result
    from IPython.display import Image, display
    import glob as gb
    for f in sorted(gb.glob('outputs/*.png')):
        display(Image(filename=f))
        break


In [ ]:
# @title 6. Save Checkpoint to Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/face2sketch'
!mkdir -p {DRIVE_DIR}/checkpoints

import glob, os
for ckpt in sorted(glob.glob('checkpoints/pix2pix_*.pt')):
    name = os.path.basename(ckpt)
    !cp -v {ckpt} {DRIVE_DIR}/checkpoints/{name}

# Also save samples
!mkdir -p {DRIVE_DIR}/samples
for sample in sorted(glob.glob('samples/*.png')):
    !cp -v {sample} {DRIVE_DIR}/samples/

print('\n✅  Saved to Google Drive: MyDrive/face2sketch/')
